# SmolVLM-256M: 4-bit GGUF on llama.cpp (CUDA)

This notebook tests a 4-bit version of SmolVLM-256M. It implements issue #1 (llama.cpp CUDA backend, GGUF quantized model).

The script does these steps:

1. It downloads the f16 language model and the f16 vision projector.
2. It makes a 4-bit model (Q4_K_M) with `llama-quantize`. It also makes Q8_0 and f16 models for comparison.
3. It runs each model on the same image and prompt with `llama-mtmd-cli`. All layers run on the GPU.

Only the language model is quantized. The vision projector stays f16 in every setup.

The script prints the size of the language model, the image encode time, the prompt time, the decode time per token, the total time, and whether the answer is the same as the f16 answer.

In [1]:
# Colab only: build llama.cpp with CUDA (about 20-30 minutes on a T4).
# -j 2: more parallel compile jobs run out of the Colab RAM (the build is killed with 'Error 137').
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --config Release -j 2 --target llama-mtmd-cli llama-quantize

fatal: destination path 'llama.cpp' already exists and is not an empty directory.
-- llama.cpp version: 0.7.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75-real CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.27.0
-- ggml commit:  9125a2e
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (0.5s)
-- Generating done (0.6s)
-- Build files have been written to: /content/llama.cpp/build
[  1%] Built target vendor-hash
[  5%] Built target ggml-base
[  5%] Built target llama-comm

In [4]:
REPO = "./edge-ai-smol-vlm-256-optimization"
BIN = "./llama.cpp/build/bin"

!python {REPO}/gguf_q4.py --image {REPO}/frame.jpg --cli {BIN}/llama-mtmd-cli --quantize {BIN}/llama-quantize

downloading SmolVLM-256M-Instruct-f16.gguf (https://huggingface.co/ggml-org/SmolVLM-256M-Instruct-GGUF/resolve/main/SmolVLM-256M-Instruct-f16.gguf)
downloading mmproj-SmolVLM-256M-Instruct-f16.gguf (https://huggingface.co/ggml-org/SmolVLM-256M-Instruct-GGUF/resolve/main/mmproj-SmolVLM-256M-Instruct-f16.gguf)
quant     LM MB encode ms prefill ms tokens decode ms/tok  total ms same as f16  answer
f16         313      99.7      145.8    9.0          4.02     281.7        True  There are no visible defects in the image.
q8_0        167      94.3      140.3    9.0          2.97     261.4        True  There are no visible defects in the image.
q4_k_m      119     100.0      172.7    9.0          3.23     301.8        True  There are no visible defects in the image.
SUMMARY {"best_quant": "q8_0", "total_ms": 261.4, "rows": [{"quant": "f16", "total_ms": 281.7, "decode_ms_per_token": 4.02, "same_as_f16": true}, {"quant": "q8_0", "total_ms": 261.4, "decode_ms_per_token": 2.97, "same_as_f16": tru

## Result: GGUF on the llama.cpp CUDA backend

The test ran on a Colab GPU with one image and a 9-token answer. Each row is the mean of 3 runs.

| Quant | Language model (MB) | Encode (ms) | Prefill (ms) | Decode (ms/token) | Total (ms) | Same answer |
|---|---|---|---|---|---|---|
| f16 | 313 | 99.7 | 145.8 | 4.02 | 281.7 | yes |
| q8_0 | 167 | 94.3 | 140.3 | 2.97 | 261.4 | yes |
| q4_k_m | 119 | 100.0 | 172.7 | 3.23 | 301.8 | yes |

- **4-bit makes the model 2.6 times smaller** (119 MB against 313 MB). The answer stays the same.
- **4-bit is not faster on this GPU.** Q4_K_M is the slowest of the three in total (302 ms). It decodes 9% slower per token than Q8_0 (3.23 against 2.97 ms), and its prefill is 23% slower (173 against 140 ms). The small model does not gain enough from the smaller weights to cover the cost of unpacking 4-bit values.
- **Q8_0 is the fastest here** (261 ms). It is about 8% faster than f16, mainly in decode.
- **The vision encoder is unchanged.** Its projector stays f16 in all three rows, so the encode time (about 95 to 100 ms) is the same.
- **Compared with the PyTorch path**, the GGUF decode is about 2 times faster per token (3 to 4 ms against 6.6 to 8 ms with CUDA graphs). The total is not directly comparable, because we did not check the number of image tiles that llama.cpp uses.
- **Three runs** give small differences. Treat differences under about 20 ms as noise.

**Conclusion for issue #1:** a 4-bit GGUF model works on the llama.cpp CUDA backend, and the answer is correct. It saves memory, not time. For speed on this GPU, use Q8_0. Q4_K_M may be worth testing again on the Jetson, where memory bandwidth is lower and the smaller weights may help more.